# Rail Pulse — 02 Live Dashboard

This notebook reads the **same SQLite database** as `01_live_collector.ipynb`.
It does not collect posts and does not run the sentiment model again.

Use the same `.venv` as Notebook 01, but Notebook 02 can use its own Jupyter kernel.

In [1]:
%pip install -q -U pandas matplotlib ipywidgets

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import sqlite3
import asyncio
from datetime import datetime, timezone

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import ipywidgets as widgets

from IPython.display import display, clear_output

DATA_DIR = Path("data")
DB = (DATA_DIR / "rail_pulse.sqlite3").resolve()

print("Notebook 02 database:", DB)
print("Database exists:", DB.exists())

if not DB.exists():
    raise FileNotFoundError(
        "rail_pulse.sqlite3 was not found. "
        "Run 01_live_collector.ipynb first from the same project folder."
    )


def connect():
    db = sqlite3.connect(
        str(DB),
        timeout=30,
        check_same_thread=False,
    )
    db.execute("PRAGMA busy_timeout=30000")
    return db

Notebook 02 database: C:\Users\ikben\Desktop\test\a\data\rail_pulse.sqlite3
Database exists: True


In [3]:
with connect() as db:
    tables = db.execute(
        "SELECT name FROM sqlite_master WHERE type='table'"
    ).fetchall()

    post_count = db.execute(
        "SELECT COUNT(*) FROM posts"
    ).fetchone()[0]

print("Database tables:", tables)
print("Posts visible to Notebook 02:", post_count)

Database tables: [('posts',), ('heartbeat',), ('sessions',), ('sqlite_sequence',), ('meta',)]
Posts visible to Notebook 02: 72


In [4]:
def load_posts(hours=48, disruptions_only=False):
    with connect() as db:
        df = pd.read_sql_query(
            """
            SELECT
                id,
                created_at,
                collected_at,
                text,
                language_hint,
                category,
                match_reason,
                matched_terms,
                disruption,
                sentiment,
                confidence,
                model
            FROM posts
            ORDER BY collected_at ASC
            """,
            db,
        )

    if df.empty:
        return df

    df["created_at"] = pd.to_datetime(
        df["created_at"], utc=True, errors="coerce"
    )
    df["collected_at"] = pd.to_datetime(
        df["collected_at"], utc=True, errors="coerce"
    )

    if hours != "All":
        cutoff = pd.Timestamp.now(tz="UTC") - pd.Timedelta(hours=int(hours))
        df = df[df["collected_at"] >= cutoff].copy()

    if disruptions_only:
        df = df[df["disruption"] == 1].copy()

    return df


def load_heartbeat():
    with connect() as db:
        return db.execute(
            """
            SELECT
                last_seen,
                events_seen,
                stored_this_session,
                reconnects,
                last_error,
                last_time_us
            FROM heartbeat
            WHERE id=1
            """
        ).fetchone()


def load_official_start():
    with connect() as db:
        row = db.execute(
            "SELECT value FROM meta WHERE key='official_start'"
        ).fetchone()
    return None if row is None else row[0]


def collector_health():
    row = load_heartbeat()

    if row is None:
        return {
            "heartbeat_age_seconds": None,
            "events_seen": 0,
            "stored_this_session": 0,
            "reconnects": 0,
            "last_error": None,
        }

    last_seen, events_seen, stored_this_session, reconnects, last_error, _ = row
    heartbeat_age = None

    try:
        dt = datetime.fromisoformat(last_seen.replace("Z", "+00:00"))
        if dt.tzinfo is None:
            dt = dt.replace(tzinfo=timezone.utc)
        heartbeat_age = (
            datetime.now(timezone.utc) - dt.astimezone(timezone.utc)
        ).total_seconds()
    except Exception:
        pass

    return {
        "heartbeat_age_seconds": heartbeat_age,
        "events_seen": events_seen,
        "stored_this_session": stored_this_session,
        "reconnects": reconnects,
        "last_error": last_error,
    }

In [5]:
window_widget = widgets.Dropdown(
    options=[
        ("Last 6 hours", 6),
        ("Last 12 hours", 12),
        ("Last 24 hours", 24),
        ("Last 48 hours", 48),
        ("Last 72 hours", 72),
        ("Last 7 days", 168),
        ("All collected data", "All"),
    ],
    value=48,
    description="Window:",
    layout=widgets.Layout(width="240px"),
)

disruption_widget = widgets.Checkbox(
    value=False,
    description="Disruption only",
)

refresh_button = widgets.Button(
    description="Refresh dashboard",
    button_style="info",
)

auto_refresh_widget = widgets.ToggleButton(
    value=False,
    description="Auto refresh 60s",
)

hour_widget = widgets.Dropdown(
    options=[],
    description="Inspect hour:",
    layout=widgets.Layout(width="700px"),
)

dashboard_output = widgets.Output()
items_output = widgets.Output()
_auto_refresh_task = None


def update_hour_options(df):
    if df.empty:
        hour_widget.options = []
        return

    temp = df.copy()
    temp["hour"] = temp["collected_at"].dt.floor("h")
    counts = temp.groupby("hour").size().sort_index(ascending=False)

    options = [
        (
            f"{hour.strftime('%Y-%m-%d %H:00 UTC')} — {count} posts",
            hour.isoformat(),
        )
        for hour, count in counts.items()
    ]

    previous = hour_widget.value
    hour_widget.options = options
    values = [value for _, value in options]

    if previous in values:
        hour_widget.value = previous
    elif options:
        hour_widget.value = options[0][1]


def refresh_dashboard(_=None):
    df = load_posts(
        hours=window_widget.value,
        disruptions_only=disruption_widget.value,
    )
    health = collector_health()
    update_hour_options(df)

    with dashboard_output:
        clear_output(wait=True)

        now = datetime.now(timezone.utc)
        print("Dashboard refreshed:", now.strftime("%Y-%m-%d %H:%M:%S UTC"))
        print("Database:", DB)
        print("Official collection start:", load_official_start())

        age = health["heartbeat_age_seconds"]

        if age is None:
            print("Collector heartbeat: unavailable")
        else:
            print(
                "Collector heartbeat age:",
                f"{age:.1f}s",
                "| events seen:",
                health["events_seen"],
                "| stored this session:",
                health["stored_this_session"],
                "| reconnects:",
                health["reconnects"],
            )

            if age <= 30:
                print("Collector status: ACTIVE")
            elif age <= 120:
                print("Collector status: heartbeat getting old")
            else:
                print("Collector status: likely stopped/disconnected")

        print("Collector last error:", health["last_error"])

        with connect() as db:
            total_all = db.execute("SELECT COUNT(*) FROM posts").fetchone()[0]
            disruption_all = db.execute(
                "SELECT COUNT(*) FROM posts WHERE disruption=1"
            ).fetchone()[0]

        print()
        print("Total stored in database:", total_all)
        print("Total disruption flagged:", disruption_all)
        print("300-item target:", "MET" if total_all >= 300 else "NOT MET")

        if df.empty:
            print()
            print("No posts for the selected filters.")
            return

        total = len(df)
        negative = int((df["sentiment"] == "negative").sum())
        neutral = int((df["sentiment"] == "neutral").sum())
        positive = int((df["sentiment"] == "positive").sum())
        disruptions = int(df["disruption"].sum())
        negative_share = negative / total * 100

        print()
        print("Selected view:", total, "posts")
        print(
            "Negative:", negative,
            "| Neutral:", neutral,
            "| Positive:", positive,
        )
        print("Disruption flagged:", disruptions)
        print("Negative share:", f"{negative_share:.1f}%")

        counts = (
            df["sentiment"]
            .value_counts()
            .reindex(["negative", "neutral", "positive"], fill_value=0)
        )

        fig, ax = plt.subplots(figsize=(8, 4))
        counts.plot(kind="bar", ax=ax)
        ax.set_title("Rail Pulse — sentiment counts")
        ax.set_xlabel("Sentiment")
        ax.set_ylabel("Number of posts")
        fig.tight_layout()
        plt.show()

        hourly = df.copy()
        hourly["hour"] = hourly["collected_at"].dt.floor("h")
        pivot = (
            hourly.groupby(["hour", "sentiment"])
            .size()
            .unstack(fill_value=0)
            .sort_index()
        )

        for label in ["negative", "neutral", "positive"]:
            if label not in pivot.columns:
                pivot[label] = 0

        fig2, ax2 = plt.subplots(figsize=(12, 5))
        for label in ["negative", "neutral", "positive"]:
            ax2.plot(pivot.index, pivot[label], marker="o", label=label)

        ax2.set_title("Rail Pulse — posts per hour by sentiment")
        ax2.set_xlabel("Time")
        ax2.set_ylabel("Number of posts")
        ax2.legend()

        if window_widget.value != "All":
            end_time = pd.Timestamp.now(tz="UTC")
            start_time = end_time - pd.Timedelta(hours=int(window_widget.value))
            ax2.set_xlim(start_time, end_time)

        ax2.xaxis.set_major_formatter(mdates.DateFormatter("%d %b\n%H:%M"))
        fig2.tight_layout()
        plt.show()

        totals = pivot.sum(axis=1).replace(0, pd.NA)
        negative_percent = (pivot["negative"] / totals * 100).fillna(0)

        fig3, ax3 = plt.subplots(figsize=(12, 4))
        ax3.plot(negative_percent.index, negative_percent.values, marker="o")
        ax3.set_title("Negative sentiment share per hour")
        ax3.set_xlabel("Time")
        ax3.set_ylabel("Negative share (%)")
        ax3.set_ylim(0, 100)
        ax3.xaxis.set_major_formatter(mdates.DateFormatter("%d %b\n%H:%M"))
        fig3.tight_layout()
        plt.show()

        print()
        print(
            "Warning: this describes collected public posts, "
            "not all Dutch train passengers."
        )


def show_selected_hour(change=None):
    with items_output:
        clear_output(wait=True)

        if not hour_widget.value:
            print("No stored hour available yet.")
            return

        selected = pd.Timestamp(hour_widget.value)
        if selected.tzinfo is None:
            selected = selected.tz_localize("UTC")
        else:
            selected = selected.tz_convert("UTC")

        df = load_posts(
            hours=window_widget.value,
            disruptions_only=disruption_widget.value,
        )

        if df.empty:
            print("No posts available.")
            return

        end = selected + pd.Timedelta(hours=1)
        subset = df[
            (df["collected_at"] >= selected)
            & (df["collected_at"] < end)
        ][
            [
                "collected_at",
                "sentiment",
                "confidence",
                "disruption",
                "matched_terms",
                "text",
            ]
        ].copy()

        print(
            "Posts behind:",
            selected.strftime("%Y-%m-%d %H:00 UTC"),
            "| Count:",
            len(subset),
        )

        if subset.empty:
            print("No posts in this hour.")
        else:
            display(subset.reset_index(drop=True))


def refresh_everything(_=None):
    refresh_dashboard()
    show_selected_hour()


refresh_button.on_click(refresh_everything)

window_widget.observe(
    lambda change: refresh_everything()
    if change["name"] == "value" else None,
    names="value",
)

disruption_widget.observe(
    lambda change: refresh_everything()
    if change["name"] == "value" else None,
    names="value",
)

hour_widget.observe(
    lambda change: show_selected_hour()
    if change["name"] == "value" else None,
    names="value",
)


async def auto_refresh_loop():
    while auto_refresh_widget.value:
        refresh_everything()
        await asyncio.sleep(60)


def auto_refresh_changed(change):
    global _auto_refresh_task

    if change["name"] != "value":
        return

    if change["new"]:
        if _auto_refresh_task is None or _auto_refresh_task.done():
            _auto_refresh_task = asyncio.create_task(auto_refresh_loop())
    else:
        print("Auto refresh stopped.")


auto_refresh_widget.observe(auto_refresh_changed, names="value")

controls = widgets.HBox(
    [
        window_widget,
        disruption_widget,
        refresh_button,
        auto_refresh_widget,
    ]
)

display(controls)
display(dashboard_output)
display(hour_widget)
display(items_output)

refresh_everything()

Output()

Dropdown(description='Inspect hour:', layout=Layout(width='700px'), options=(), value=None)

Output()

## How to use it

- **Refresh dashboard** updates immediately.
- **Auto refresh 60s** refreshes without blocking the kernel.
- The `Dashboard refreshed:` timestamp proves a refresh happened even if no new posts arrived.
- **Disruption only** filters to disruption-flagged posts.
- **Inspect hour** shows the posts behind that hour in the graph.

Notebook 01 must remain running separately for new posts to arrive.